In [ ]:
import os
import json
import pandas as pd
import re
import numpy as np


# Connect to Google Drive
import gspread
import gspread_dataframe
from google.oauth2.service_account import Credentials
from google.oauth2 import service_account
from googleapiclient.discovery import build
from gspread_dataframe import set_with_dataframe
from gspread_dataframe import get_as_dataframe

In [ ]:
# 1. Fetch credentials from environment variable
creds_env = os.environ.get("GDRIVE_CREDENTIALS_KC")

if not creds_env:
    raise ValueError("Environment variable 'GDRIVE_CREDENTIALS' was not found.")

creds_json = json.loads(creds_env)

# 2. Define required scopes
scopes = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]

# 3. Authenticate service account
creds = service_account.Credentials.from_service_account_info(
    creds_json, scopes=scopes
)

# 4. Initialize Google API clients
drive_service = build("drive", "v3", credentials=creds)
sheets_service = build("sheets", "v4", credentials=creds)

gc = gspread.authorize(creds)

print("Google Drive and Sheets services successfully initialized.")

Mounted at /content/drive


In [ ]:
# Open files

# Open organic posts file
forms_data = gc.open_by_key('1VK7_oyA3boJaudPaAiwk7xYl6sxReed63eOYBP9ahxo')
forms_data = forms_data.get_worksheet(0)
forms_data = get_as_dataframe(forms_data)

influencers_posts = gc.open_by_key('1S86wWk2yO525qC0JQ6IZ6G5gFCYhzzn4Ny6TwZC2E98')
influencers_posts = influencers_posts.get_worksheet(0)
influencers_posts = get_as_dataframe(influencers_posts)

influencers_comments = gc.open_by_key('1orR6-MXGNajad6q5IP1dakAPQMM5lyUMX-718YbQzhI')
influencers_comments = influencers_comments.get_worksheet(0)
influencers_comments = get_as_dataframe(influencers_comments)

posts_data = gc.open_by_key('1TvqNwg2GeCpBRNKBw87-qcbsG1yK2BaI8UQEbbZgCBw')
posts_data = posts_data.worksheet('data_profile_post_max')
posts_data = get_as_dataframe(posts_data)

posts_comments = gc.open_by_key('1dD69AANExCtYQG0g3MX8q5Sv794J0ajkRJ2GLGhaqLI')
posts_comments = posts_comments.get_worksheet(0)
posts_comments = get_as_dataframe(posts_comments)




In [ ]:
# Clean databases
forms_data = forms_data.fillna(0)
influencers_posts = influencers_posts.fillna(0)
influencers_comments = influencers_comments.fillna(0)
posts_data = posts_data.fillna(0)
posts_comments = posts_comments.fillna(0)

/tmp/ipykernel_1269/135062332.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  influencers_comments = influencers_comments.fillna(0)
/tmp/ipykernel_1269/135062332.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  posts_comments = posts_comments.fillna(0)


In [ ]:
# Create influencers baseline for negative sentiment
neg_sent_baselines = influencers_comments.groupby('perfil').agg(
    detrator_count=('sentimento_nps', lambda x: (x == 'detrator').sum()),
    total_rows=('sentimento_nps', 'count')
).reset_index()


neg_sent_baselines['negative_influencer_baseline'] = neg_sent_baselines['detrator_count'] / neg_sent_baselines['total_rows']


In [ ]:
neg_sent_baselines

,perfil,detrator_count,total_rows,negative_influencer_baseline
0,Mayicrystal,0,21,0.000000
1,Tamayoalejandra,0,26,0.000000
2,___lauraortega,3,46,0.065217
3,_analyespinoza,0,29,0.000000
4,_kiara.c_,0,2,0.000000
...,...,...,...,...
180,vic_saravia,10,238,0.042017
181,vickylsmakeup,18,117,0.153846
182,yazminvasquezpuali,80,487,0.164271
183,yos_daza,2,12,0.166667


In [ ]:
# Create influencers baseline for engagement rate
# 1. Create the "total_interactions" column
influencers_posts['total_interactions'] = influencers_posts['comment_count'] + influencers_posts['like_count']

# 2. Remove rows where play_count is 0
influencers_posts = influencers_posts[influencers_posts['play_count'] != 0]

# 3. Aggregate (sum) total_interactions and play_count by username
er_baselines = influencers_posts.groupby('username').agg(
    total_interactions=('total_interactions', 'sum'),
    play_count=('play_count', 'sum')
).reset_index()

# 4. Create the "er_influencer_baseline" column
er_baselines['er_influencer_baseline'] = (
    er_baselines['total_interactions'] / er_baselines['play_count']
)

In [ ]:
er_baselines

,username,total_interactions,play_count,er_influencer_baseline
0,Mayicrystal,687.0,22111.0,0.031071
1,Tamayoalejandra,7095.0,169919.0,0.041755
2,___lauraortega,17944.0,870256.0,0.020619
3,_analyespinoza,669.0,12662.0,0.052835
4,_kiara.c_,8.0,412.0,0.019417
...,...,...,...,...
177,vic_saravia,1511.0,5115489.0,0.000295
178,vickylsmakeup,61521.0,4531222.0,0.013577
179,yazminvasquezpuali,1596156.0,77737772.0,0.020533
180,yos_daza,586.0,7538.0,0.077739


In [ ]:
# Estimate sentimment for each published post
sentiment_posts = posts_comments.groupby(['post_url', 'perfil']).agg(
    positive_count=('sentimento_nps', lambda x: (x == 'promotor').sum()),
    negative_count=('sentimento_nps', lambda x: (x == 'detrator').sum()),
    neutral_count=('sentimento_nps', lambda x: (x == 'neutro').sum())
).reset_index()

# 2. Sum the three sentiment columns to get the total
sentiment_posts['total_sentiments'] = (
    sentiment_posts['positive_count'] +
    sentiment_posts['negative_count'] +
    sentiment_posts['neutral_count']
)

# 3. Calculate the percentage of each sentiment over the total
sentiment_posts['positive_percentage'] = sentiment_posts['positive_count'] / sentiment_posts['total_sentiments']
sentiment_posts['negative_percentage'] = sentiment_posts['negative_count'] / sentiment_posts['total_sentiments']
sentiment_posts['neutral_percentage'] = sentiment_posts['neutral_count'] / sentiment_posts['total_sentiments']

In [ ]:
sentiment_posts

,post_url,perfil,positive_count,negative_count,neutral_count,total_sentiments,positive_percentage,negative_percentage,neutral_percentage
0,google.com/url?q=https://www.instagram.com/ree...,beautyh_aleja,1,3,0,4,0.250000,0.750000,0.000000
1,https://www.instagram.com/p/DIxMPhDN-m7,agucasanova,9,1,0,10,0.900000,0.100000,0.000000
2,https://www.instagram.com/p/DMyYgeeSekO,camugomezares,7,0,0,7,1.000000,0.000000,0.000000
3,https://www.instagram.com/p/DNa49BOOWuD,vic_saravia,6,0,0,6,1.000000,0.000000,0.000000
4,https://www.instagram.com/p/DNyoDIl0qh3,darioorsi,6,3,0,9,0.666667,0.333333,0.000000
...,...,...,...,...,...,...,...,...,...
307,https://www.instagram.com/reels/DagnKA7JGyt/,dianabenitezsv,4,0,0,4,1.000000,0.000000,0.000000
308,https://www.instagram.com/reels/DaqxzIbgPbh/,missali_b,3,0,0,3,1.000000,0.000000,0.000000
309,https://www.instagram.com/reels/DbMbvYPu1g2/,renata_bravo_,2,3,3,8,0.250000,0.375000,0.375000
310,https://www.instagram.com/reels/DbMkz0NJgCJ/,eligbarahona,6,1,1,8,0.750000,0.125000,0.125000


In [ ]:
# Prepare datasets for join

posts_data = posts_data.rename(columns={'username_shared': 'influencer'})
forms_data = forms_data.rename(columns={'Link of Post': 'url'})
sentiment_posts = sentiment_posts.rename(columns={'post_url': 'url'})
neg_sent_baselines = neg_sent_baselines.rename(columns={'perfil': 'influencer'})
er_baselines = er_baselines.rename(columns={'username': 'influencer'})

In [ ]:
# Join datasets
instagram_influencers_final = posts_data
instagram_influencers_final = instagram_influencers_final.merge(forms_data, on='url', how='left')
instagram_influencers_final = instagram_influencers_final.merge(sentiment_posts, on='url', how='left')
instagram_influencers_final = instagram_influencers_final.merge(neg_sent_baselines, on='influencer', how='left')
instagram_influencers_final = instagram_influencers_final.merge(er_baselines, on='influencer', how='left')
instagram_influencers_final

,run_datetime,Plataform_x,username,influencer,followers_count,following_count,total_posts_count,code,taken_at,url,...,total_sentiments,positive_percentage,negative_percentage,neutral_percentage,detrator_count,total_rows,negative_influencer_baseline,total_interactions,play_count_y,er_influencer_baseline
0,2026-07-30 11:03:12,Instagram,rusita.bonita,rusita.bonita,39098.0,162.0,279.0,DZyLoJ_uR2k,2026-06-19 19:24:20,https://www.instagram.com/reel/DZyLoJ_uR2k/,...,3.0,0.666667,0.000000,0.333333,0.0,30.0,0.000000,1055.0,20980.0,0.050286
1,2026-07-30 17:58:45,Instagram,ariaguep,ariaguep,1061.0,433.0,159.0,DZYkZ8FRlwm,2026-06-09 20:39:54,https://www.instagram.com/reels/DZYkZ8FRlwm/,...,10.0,0.900000,0.100000,0.000000,0.0,29.0,0.000000,329.0,7869.0,0.041810
2,2026-07-30 11:03:36,Instagram,soydadyta,soydadyta,10368.0,3059.0,620.0,DZxniNBBZFs,2026-06-19 14:08:14,https://www.instagram.com/reels/DZxniNBBZFs/,...,5.0,1.000000,0.000000,0.000000,1.0,21.0,0.047619,30126.0,946910.0,0.031815
3,2026-07-30 11:00:22,Instagram,tomastudelach,valeryrevello,378639.0,1254.0,1168.0,DZxbrZPO4q3,2026-06-19 12:25:28,https://www.instagram.com/reel/DZxbrZPO4q3/,...,10.0,1.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-07-30 11:01:23,Instagram,dr.somocurcio,dr.somocurcio,62884.0,8023.0,1263.0,DZx4YOUom_d,2026-06-19 16:45:35,https://www.instagram.com/p/DZx4YOUom_d/%20%20...,...,4.0,0.500000,0.000000,0.500000,5.0,28.0,0.178571,940.0,27695.0,0.033941
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
308,2026-07-30 17:52:27,Instagram,analudada,analudada,339377.0,2904.0,1000.0,Da3-wzmOc6K,2026-07-16 21:59:49,https://www.instagram.com/reel/Da3-wzmOc6K/,...,6.0,0.833333,0.166667,0.000000,1.0,46.0,0.021739,23463.0,459743.0,0.051035
309,2026-07-30 10:48:00,Instagram,Mayicrystal,mayicrystal,24633.0,2663.0,2213.0,Da0F9ASx9ac,2026-07-15 9:52:58,https://www.instagram.com/p/Da0F9ASx9ac/,...,15.0,0.800000,0.000000,0.200000,NaN,NaN,NaN,NaN,NaN,NaN
310,2026-08-07 3:04:41,Instagram,___lauraortega,___lauraortega,348323.0,1173.0,278.0,DbR4rV9xy82,2026-07-26 23:30:16,https://www.instagram.com/reel/DbR4rV9xy82/?ig...,...,10.0,0.700000,0.000000,0.300000,3.0,46.0,0.065217,17944.0,870256.0,0.020619
311,2026-08-07 3:04:51,Instagram,estefaniaacuna,estefaniaacuna,103274.0,1255.0,2646.0,DbGwaNLurkG,2026-07-22 15:42:46,https://www.instagram.com/reel/DbGwaNLurkG/?ut...,...,0.0,NaN,NaN,NaN,4.0,44.0,0.090909,34950.0,1009859.0,0.034609


In [ ]:
# Agregar código orgánico
instagram_influencers_final["Organic_ID"] = instagram_influencers_final["code"]


In [ ]:
# Rename columns
instagram_influencers_final = instagram_influencers_final.rename(columns={'post_caption': 'copy'})
instagram_influencers_final = instagram_influencers_final.rename(columns={'Plataform_x': 'platform'})
instagram_influencers_final = instagram_influencers_final.rename(columns={'taken_at': 'date_published'})

instagram_influencers_final = instagram_influencers_final.rename(columns={'media_type': 'format'})
instagram_influencers_final = instagram_influencers_final.rename(columns={'play_count_x': 'views'})
instagram_influencers_final = instagram_influencers_final.rename(columns={'comment_count': 'comments'})
instagram_influencers_final = instagram_influencers_final.rename(columns={'like_count': 'likes'})
instagram_influencers_final = instagram_influencers_final.rename(columns={'Marca': 'brand'})


In [ ]:
# Crear columnas
instagram_influencers_final["shares"] = 0
instagram_influencers_final["saves"] = 0
instagram_influencers_final["content_type"] = "Influencers"
instagram_influencers_final["total_interactions"] = instagram_influencers_final["likes"] + instagram_influencers_final["comments"] + instagram_influencers_final["shares"] + instagram_influencers_final["saves"]
instagram_influencers_final["engagement_rate"] = (
    instagram_influencers_final["total_interactions"]
    .div(instagram_influencers_final["views"])
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)
instagram_influencers_final["positive_comments"] = instagram_influencers_final["positive_percentage"] * instagram_influencers_final["comments"]
instagram_influencers_final["negative_comments"] = instagram_influencers_final["negative_percentage"] * instagram_influencers_final["comments"]
instagram_influencers_final["neutral_comments"] = instagram_influencers_final["neutral_percentage"] * instagram_influencers_final["comments"]
instagram_influencers_final["delta_eng_rate"] = instagram_influencers_final["engagement_rate"] - instagram_influencers_final["er_influencer_baseline"]
instagram_influencers_final["delta_neg_sentiment"] = instagram_influencers_final["negative_percentage"] - instagram_influencers_final["negative_influencer_baseline"]



In [ ]:
# Apply boosting criteria

def classify_boost(row):
    if row['delta_eng_rate'] > 0 and row['delta_neg_sentiment'] <= 0:
        return 'Boost'
    elif row['delta_eng_rate'] > 0 and row['delta_neg_sentiment'] > 0:
        return 'Review'
    else:
        return 'No Boost'

instagram_influencers_final['Accionable'] = instagram_influencers_final.apply(classify_boost, axis=1)

In [ ]:
# Seleccionar y ordenar columnas

cols = [
    "url",
    "copy",
    "date_published",
    "platform",
    "format",
    "influencer",
    "Country",
    "Organic_ID",
    "brand",
    "content_type",
    "views",
    "likes",
    "comments",
    "shares",
    "saves",
    "total_interactions",
    "engagement_rate",
    "positive_percentage",
    "negative_percentage",
    "neutral_percentage",
    "positive_comments",
    "negative_comments",
    "neutral_comments",
    "er_influencer_baseline",
    "negative_influencer_baseline",
    "Accionable",
    "run_datetime"
]

instagram_influencers_final = instagram_influencers_final[cols]
instagram_influencers_final = instagram_influencers_final.reindex(columns=cols)

In [ ]:
# Adjust date
instagram_influencers_final["date_published"] = pd.to_datetime(
    instagram_influencers_final["date_published"],
    format="mixed",
    errors="coerce",
).dt.date

instagram_influencers_final["run_datetime"] = pd.to_datetime(
    instagram_influencers_final["run_datetime"],
    format="mixed",
    errors="coerce",
).dt.date

In [ ]:
# Force missing likes (-1) to 0
instagram_influencers_final["likes"] = instagram_influencers_final["likes"].clip(lower=0)

In [ ]:
# Save final table
# Open the destination sheets file
sh = gc.open_by_key('1s9zK9LsAGnkEynPtM_EiPZ44UYhate8QuDZBmu0MjP0')
worksheet = sh.get_worksheet(0)

# Replace old data with new data
set_with_dataframe(worksheet, instagram_influencers_final)
print("DataFrame saved successfully!")

DataFrame saved successfully!
